In [ ]:
import json
import re
from pathlib import Path
from typing import Dict, List

import cv2
import numpy as np
import pandas as pd
import pytesseract

print('OpenCV version:', cv2.__version__)

In [ ]:
ROOT = Path('.').resolve()
EXAMPLES_CSV = ROOT / 'data' / 'ocr_identity_examples.csv'
REPORT_PATH = ROOT / 'reports' / 'tesseract_report.json'

TESS_LANG = 'eng'
TESS_PSM = 6
# If needed on Windows, uncomment and set executable path:
# pytesseract.pytesseract.tesseract_cmd = r'C:\\Program Files\\Tesseract-OCR\\tesseract.exe'

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)
print('CSV:', EXAMPLES_CSV)

In [ ]:
TRUE_VALUES = {'1', 'true', 'yes', 'y', 'same', 'match', 'positive'}

def parse_bool_label(raw: str) -> bool:
    return str(raw).strip().lower() in TRUE_VALUES

def safe_div(n: float, d: float) -> float:
    return 0.0 if d == 0 else n / d

def normalize_identifier(value: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', str(value).upper())

def compact_preview(text: str, max_chars: int = 220) -> str:
    t = ' '.join(text.split())
    return t if len(t) <= max_chars else t[:max_chars - 3].rstrip() + '...'

def preprocess_for_ocr(image_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    denoised = cv2.bilateralFilter(gray, 9, 75, 75)
    scaled = cv2.resize(denoised, None, fx=1.8, fy=1.8, interpolation=cv2.INTER_CUBIC)
    thresh = cv2.adaptiveThreshold(
        scaled, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
    )
    return thresh

def extract_candidates(text: str, doc_type: str) -> List[str]:
    dt = (doc_type or 'generic').strip().lower()

    pattern_map = {
        'cnic': [
            r'\b\d{5}-?\d{7}-?\d\b',
            r'\b\d{13}\b',
        ],
        'driving_license': [
            r'\b[A-Z]{2,5}-?\d{3,12}\b',
            r'\b\d{1,4}-?[A-Z]{1,4}-?\d{2,10}\b',
            r'\b[A-Z]{2,5}\d{4,12}\b',
        ],
        'generic': [
            r'\b\d{5}-?\d{7}-?\d\b',
            r'\b\d{13}\b',
            r'\b[A-Z]{2,5}-?\d{3,12}\b',
            r'\b\d{1,4}-?[A-Z]{1,4}-?\d{2,10}\b',
        ],
    }

    patterns = pattern_map.get(dt, pattern_map['generic'])

    found = []
    for pat in patterns:
        for m in re.findall(pat, text, flags=re.IGNORECASE):
            value = str(m).strip().upper()
            if dt == 'cnic' and re.fullmatch(r'\d{13}', value):
                value = f'{value[:5]}-{value[5:12]}-{value[12]}'
            found.append(value)

    deduped = []
    seen = set()
    for x in found:
        key = normalize_identifier(x)
        if not key or key in seen:
            continue
        seen.add(key)
        deduped.append(x)
    return deduped

def run_ocr(image_path: Path, lang: str = 'eng', psm: int = 6) -> Dict:
    if not image_path.exists():
        raise FileNotFoundError(f'Image not found: {image_path}')

    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f'Failed to read image: {image_path}')

    proc = preprocess_for_ocr(img)
    config = f'--oem 3 --psm {psm}'
    text = pytesseract.image_to_string(proc, lang=lang, config=config)

    return {
        'text': text,
        'text_preview': compact_preview(text),
        'config': config,
        'lang': lang,
    }

def verify_identity(image_path: Path, claimed_number: str, doc_type: str, lang: str = 'eng', psm: int = 6) -> Dict:
    ocr = run_ocr(image_path=image_path, lang=lang, psm=psm)
    candidates = extract_candidates(ocr['text'], doc_type)
    extracted = candidates[0] if candidates else ''

    expected_norm = normalize_identifier(claimed_number)
    extracted_norm = normalize_identifier(extracted)
    is_match = bool(expected_norm and extracted_norm and expected_norm == extracted_norm)

    if is_match:
        reason = 'match'
    elif not extracted_norm:
        reason = 'number_not_found'
    else:
        reason = 'mismatch'

    return {
        'image_path': str(image_path),
        'doc_type': doc_type,
        'claimed_number': claimed_number,
        'extracted_number': extracted,
        'candidates': candidates,
        'match': is_match,
        'reason': reason,
        'text_preview': ocr['text_preview'],
        'ocr_engine': 'Tesseract',
        'lang': lang,
        'psm': psm,
    }

In [ ]:
if not EXAMPLES_CSV.exists():
    raise FileNotFoundError(f'Missing CSV: {EXAMPLES_CSV}')

df = pd.read_csv(EXAMPLES_CSV)
required_cols = {'image_path', 'claimed_number', 'label'}
missing = required_cols.difference(df.columns)
if missing:
    raise ValueError(f'CSV missing required columns: {sorted(missing)}')

if 'doc_type' not in df.columns:
    df['doc_type'] = 'generic'

df['actual_match'] = df['label'].astype(str).map(parse_bool_label)
df.head()

In [ ]:
tp = tn = fp = fn = 0
processed = 0
skipped = 0
rows = []

for _, row in df.iterrows():
    image_path = Path(str(row['image_path']))
    claimed = str(row['claimed_number'])
    doc_type = str(row['doc_type'])
    actual = bool(row['actual_match'])

    try:
        out = verify_identity(image_path=image_path, claimed_number=claimed, doc_type=doc_type, lang=TESS_LANG, psm=TESS_PSM)
        pred = bool(out['match'])

        if actual and pred:
            tp += 1
        elif (not actual) and (not pred):
            tn += 1
        elif (not actual) and pred:
            fp += 1
        else:
            fn += 1

        rows.append({
            'image_path': out['image_path'],
            'doc_type': out['doc_type'],
            'claimed_number': out['claimed_number'],
            'extracted_number': out['extracted_number'],
            'actual_match': actual,
            'predicted_match': pred,
            'reason': out['reason'],
            'text_preview': out['text_preview'],
            'error': ''
        })
        processed += 1
    except Exception as exc:
        rows.append({
            'image_path': str(image_path),
            'doc_type': doc_type,
            'claimed_number': claimed,
            'extracted_number': '',
            'actual_match': actual,
            'predicted_match': None,
            'reason': 'error',
            'text_preview': '',
            'error': str(exc),
        })
        skipped += 1

total = tp + tn + fp + fn
accuracy = safe_div(tp + tn, total)
precision = safe_div(tp, tp + fp)
recall = safe_div(tp, tp + fn)
f1 = safe_div(2 * precision * recall, precision + recall)

summary = {
    'processed': int(processed),
    'skipped': int(skipped),
    'tp': int(tp),
    'tn': int(tn),
    'fp': int(fp),
    'fn': int(fn),
    'accuracy': float(round(accuracy, 6)),
    'precision': float(round(precision, 6)),
    'recall': float(round(recall, 6)),
    'f1': float(round(f1, 6)),
    'ocr_engine': 'Tesseract',
    'lang': TESS_LANG,
    'psm': TESS_PSM,
}

summary

In [ ]:
report = {
    'model_info': {
        'ocr_engine': 'Tesseract',
        'lang': TESS_LANG,
        'psm': TESS_PSM,
    },
    'dataset': {
        'examples_csv': str(EXAMPLES_CSV),
        'rows': int(len(df)),
    },
    'summary': summary,
    'predictions': rows,
}

REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved report to:', REPORT_PATH)

In [ ]:
# sample = verify_identity(
#     image_path=Path('./data/docs/cnic_clear_1.jpg'),
#     claimed_number='12345-1234567-1',
#     doc_type='cnic',
#     lang=TESS_LANG,
#     psm=TESS_PSM,
# )
# sample

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import SequenceMatcher

viz_df = pd.DataFrame(rows)
valid_df = viz_df[(viz_df['error'] == '') & viz_df['predicted_match'].notna()].copy()

if valid_df.empty:
    raise ValueError('No valid prediction rows available for plotting. Check data paths and rerun previous cells.')

def identifier_similarity(claimed: str, extracted: str) -> float:
    a = normalize_identifier(claimed)
    b = normalize_identifier(extracted)
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    return float(SequenceMatcher(None, a, b).ratio())

valid_df['score'] = valid_df.apply(
    lambda r: identifier_similarity(r['claimed_number'], r['extracted_number']),
    axis=1,
)

y_true = valid_df['actual_match'].astype(bool).to_numpy()
y_pred = valid_df['predicted_match'].astype(bool).to_numpy()
scores = valid_df['score'].astype(float).to_numpy()

# --- Confusion Matrix ---
cm = np.array([
    [np.sum((~y_true) & (~y_pred)), np.sum((~y_true) & (y_pred))],
    [np.sum((y_true) & (~y_pred)), np.sum((y_true) & (y_pred))],
], dtype=int)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=['Pred: Mismatch', 'Pred: Match'],
    yticklabels=['Actual: Mismatch', 'Actual: Match'],
)
plt.title('Tesseract Identity Verification Confusion Matrix')
plt.tight_layout()
plt.show()

# --- ROC Curve from similarity score ---
thresholds = np.unique(scores)
if thresholds.size == 1:
    thresholds = np.array([thresholds[0] - 1e-6, thresholds[0], thresholds[0] + 1e-6])

thresholds = np.concatenate(([scores.max() + 1e-6], np.sort(thresholds)[::-1], [scores.min() - 1e-6]))

roc_points = []
for t in thresholds:
    yp = scores >= t
    tp = np.sum((y_true) & (yp))
    tn = np.sum((~y_true) & (~yp))
    fp = np.sum((~y_true) & (yp))
    fn = np.sum((y_true) & (~yp))

    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    roc_points.append((fpr, tpr))

roc_points = sorted(roc_points, key=lambda x: x[0])
fpr_vals = np.array([p[0] for p in roc_points])
tpr_vals = np.array([p[1] for p in roc_points])
auc = float(np.trapz(tpr_vals, fpr_vals))

plt.figure(figsize=(6, 5))
plt.plot(fpr_vals, tpr_vals, label=f'Tesseract ROC (AUC={auc:.4f})', color='tab:green')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Identity Verification ROC Curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Valid rows used for plotting: {len(valid_df)}')
print(f'ROC AUC: {auc:.6f}')

## Advanced Evaluation: Threshold Diagnostics, Uncertainty, and Failure Patterns

This section strengthens the OCR notebook with:
- similarity-threshold operating-point analysis
- bootstrap confidence intervals for key metrics
- error breakdown by document type and difficult examples

In [ ]:
from numpy.random import default_rng


def _f1_from_counts(tp: int, fp: int, fn: int) -> float:
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    return safe_div(2 * precision * recall, precision + recall)


def _roc_auc_manual(y_true_bool: np.ndarray, score_vals: np.ndarray) -> float:
    thresholds_local = np.unique(score_vals)
    if thresholds_local.size == 1:
        thresholds_local = np.array([thresholds_local[0] - 1e-6, thresholds_local[0], thresholds_local[0] + 1e-6])

    thresholds_local = np.concatenate(
        ([score_vals.max() + 1e-6], np.sort(thresholds_local)[::-1], [score_vals.min() - 1e-6])
    )

    pts = []
    for t in thresholds_local:
        yp = score_vals >= t
        tp = np.sum((y_true_bool) & (yp))
        tn = np.sum((~y_true_bool) & (~yp))
        fp = np.sum((~y_true_bool) & (yp))
        fn = np.sum((y_true_bool) & (~yp))
        tpr = safe_div(tp, tp + fn)
        fpr = safe_div(fp, fp + tn)
        pts.append((fpr, tpr))

    pts = sorted(pts, key=lambda x: x[0])
    fpr_vals = np.array([p[0] for p in pts])
    tpr_vals = np.array([p[1] for p in pts])
    return float(np.trapz(tpr_vals, fpr_vals))


def _bootstrap_ci(values: np.ndarray, alpha: float = 0.05) -> tuple:
    clean = values[np.isfinite(values)]
    if clean.size == 0:
        return (float('nan'), float('nan'), float('nan'))
    return (
        float(np.quantile(clean, alpha / 2)),
        float(np.quantile(clean, 0.5)),
        float(np.quantile(clean, 1 - alpha / 2)),
    )


# --- Operating point analysis based on identifier similarity ---
threshold_grid = np.round(np.linspace(0.0, 1.0, 51), 2)
op_rows = []

for t in threshold_grid:
    yp = scores >= t
    tp = int(np.sum((y_true) & (yp)))
    tn = int(np.sum((~y_true) & (~yp)))
    fp = int(np.sum((~y_true) & (yp)))
    fn = int(np.sum((y_true) & (~yp)))

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)
    acc = safe_div(tp + tn, tp + tn + fp + fn)
    fpr = safe_div(fp, fp + tn)

    op_rows.append(
        {
            'threshold': float(t),
            'accuracy': float(acc),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1),
            'fpr': float(fpr),
            'youden_j': float(recall - fpr),
        }
    )

op_df = pd.DataFrame(op_rows)
best_by_f1 = op_df.sort_values(['f1', 'accuracy'], ascending=False).iloc[0]
best_by_youden = op_df.sort_values(['youden_j', 'accuracy'], ascending=False).iloc[0]

print('Top thresholds by F1')
print(op_df.sort_values(['f1', 'accuracy'], ascending=False).head(5).to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('\nTop thresholds by Youden J')
print(op_df.sort_values(['youden_j', 'accuracy'], ascending=False).head(5).to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print(f"\nBest threshold by F1: {best_by_f1['threshold']:.2f}")
print(f"Best threshold by Youden J: {best_by_youden['threshold']:.2f}")

# --- Bootstrap confidence intervals for exact-match rule and tuned-score rule ---
rng = default_rng(42)
n = len(y_true)
exact_pred = y_pred.copy()
tuned_threshold = float(best_by_f1['threshold'])
tuned_pred = scores >= tuned_threshold

metrics = {
    'exact_rule': {'acc': [], 'f1': [], 'auc': []},
    'tuned_score_rule': {'acc': [], 'f1': [], 'auc': []},
}

for _ in range(1000):
    idx = rng.integers(0, n, n)
    yt = y_true[idx]
    sc = scores[idx]

    for name, pred_source in [('exact_rule', exact_pred), ('tuned_score_rule', tuned_pred)]:
        yp = pred_source[idx]
        tp = int(np.sum((yt) & (yp)))
        tn = int(np.sum((~yt) & (~yp)))
        fp = int(np.sum((~yt) & (yp)))
        fn = int(np.sum((yt) & (~yp)))

        metrics[name]['acc'].append(safe_div(tp + tn, tp + tn + fp + fn))
        metrics[name]['f1'].append(_f1_from_counts(tp, fp, fn))

        if np.unique(yt).size > 1:
            metrics[name]['auc'].append(_roc_auc_manual(yt, sc))

ci_rows = []
for method_name, vals in metrics.items():
    for metric_name, arr in vals.items():
        lower, med, upper = _bootstrap_ci(np.array(arr))
        ci_rows.append(
            {
                'method': method_name,
                'metric': metric_name,
                'ci_95_lower': lower,
                'median': med,
                'ci_95_upper': upper,
            }
        )

ci_df = pd.DataFrame(ci_rows)
print('\nBootstrap 95% confidence intervals (n=1000)')
print(ci_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# --- Error analysis by document type and hard cases ---
analysis_df = valid_df.copy()
analysis_df['error_type'] = np.where(
    (analysis_df['actual_match'] == False) & (analysis_df['predicted_match'] == True),
    'false_positive',
    np.where(
        (analysis_df['actual_match'] == True) & (analysis_df['predicted_match'] == False),
        'false_negative',
        'correct',
    ),
)

if 'doc_type' in analysis_df.columns:
    per_doc = []
    for doc_type, g in analysis_df.groupby('doc_type'):
        yt = g['actual_match'].astype(bool).to_numpy()
        yp = g['predicted_match'].astype(bool).to_numpy()
        tp = int(np.sum((yt) & (yp)))
        tn = int(np.sum((~yt) & (~yp)))
        fp = int(np.sum((~yt) & (yp)))
        fn = int(np.sum((yt) & (~yp)))
        per_doc.append(
            {
                'doc_type': doc_type,
                'count': len(g),
                'accuracy': safe_div(tp + tn, tp + tn + fp + fn),
                'precision': safe_div(tp, tp + fp),
                'recall': safe_div(tp, tp + fn),
                'f1': _f1_from_counts(tp, fp, fn),
            }
        )

    per_doc_df = pd.DataFrame(per_doc).sort_values('f1', ascending=False)
    print('\nPerformance by doc_type')
    print(per_doc_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

errors_only = analysis_df[analysis_df['error_type'] != 'correct'].copy()
print(f'\nTotal errors: {len(errors_only)} / {len(analysis_df)}')
if not errors_only.empty:
    print(errors_only['error_type'].value_counts().to_string())

    fp_top = errors_only[errors_only['error_type'] == 'false_positive'].sort_values('score', ascending=False).head(8)
    fn_top = errors_only[errors_only['error_type'] == 'false_negative'].sort_values('score', ascending=True).head(8)

    if not fp_top.empty:
        print('\nMost confident false positives:')
        print(fp_top[['image_path', 'doc_type', 'claimed_number', 'extracted_number', 'score']].to_string(index=False))

    if not fn_top.empty:
        print('\nMost severe false negatives:')
        print(fn_top[['image_path', 'doc_type', 'claimed_number', 'extracted_number', 'score']].to_string(index=False))